[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/dif/u4_derivada.ipynb)

# Dif. U4 · La derivada
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 4 del bloque de cálculo diferencial. Tiene una sección por subtema, en el mismo orden que el libro:

1. Pendiente de una recta y pendiente de la secante
2. Paso al límite: definición de derivada
3. Primeros ejemplos: $x^2$, $x^3$, $1/x$, $\sqrt{x}$
4. Notaciones $f'(x)$ y $dy/dx$; recta tangente
5. La derivada como razón de cambio
6. Propiedades: derivable implica continua; puntos no derivables

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección termina con una celda **Contrasta**: antes de ejecutarla, calcula a mano el caso que se indica y escribe tu resultado. En la definición de derivada, $h$ es una variable que tiende a cero, nunca un número fijo: si tu resultado todavía depende de $h$, falta el paso al límite.

**Etiqueta del repositorio.** `DIF-U4` (bloque, unidad). Las celdas de código de cada sección vienen de `dif/u4_derivada/NN_*.py`; los fragmentos que el libro imprime, de `dif/u4_derivada/fragmentos/`.

**Convenciones.** Punto decimal. Los valores se *redondean* (no se truncan) a las cifras significativas que elijas. Para escribir expresiones usa `x` como variable (también cuando represente el tiempo), `*` para multiplicar, `**` o `^` para potencias, `sqrt(...)` para raíces cuadradas, `cbrt(...)` para la raíz cúbica real, `abs(...)` para el valor absoluto, `exp(...)` para $e^x$ y `log(...)` para el logaritmo natural. Los ángulos de `sin`, `cos` y `tan` van en **radianes**.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor, rationalize)
from sympy.calculus.util import continuous_domain, function_range

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Símbolos y utilidades compartidas por todo el notebook
x = sp.symbols("x", real=True)
h = sp.symbols("h", real=True)     # incremento: variable que tiende a cero

_LOCAL = {"x": x, "sqrt": sp.sqrt, "abs": sp.Abs, "exp": sp.exp, "log": sp.log,
          "cbrt": lambda e: sp.sign(e) * sp.Abs(e)**sp.Rational(1, 3),
          "pi": sp.pi, "sin": sp.sin, "cos": sp.cos, "tan": sp.tan, "E": sp.E}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Symbol": sp.Symbol, "Function": sp.Function}
# rationalize: 0.1 se lee como 1/10, para que los límites salgan exactos
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor, rationalize)


def parsear(texto):
    """Convierte un texto en expresión de sympy en la variable x, o explica qué salió mal."""
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe una expresión, por ejemplo: (x**3 - 1)/(x - 1)")
    desconocidos = sorted(set(re.findall(r"[A-Za-z_]\w*", texto)) - set(_LOCAL))
    if desconocidos:
        raise ValueError(f"No reconozco {desconocidos}. Usa x como variable y, si hace falta, "
                         "sqrt, cbrt, abs, exp, log (natural), sin, cos, tan o pi.")
    try:
        expr = parse_expr(texto, local_dict=_LOCAL, global_dict=dict(_GLOBAL),
                          transformations=_TRANSF)
    except Exception as err:
        raise ValueError("No pude leer la expresión. Usa x como variable, * para multiplicar, "
                         "** o ^ para potencias y sqrt(...) para raíces.") from err
    if expr.atoms(sp.core.function.AppliedUndef):
        raise ValueError("Hay una función que no conozco. Las permitidas son sqrt, cbrt, abs, exp, log, sin, cos y tan.")
    libres = {str(s) for s in expr.free_symbols} - {"x"}
    if libres:
        raise ValueError(f"Símbolos no permitidos: {sorted(libres)}. Solo puedes usar x.")
    return expr


def numero(texto):
    """Lee un número escrito como 2, -0.5, 1/3, pi/2 o sqrt(2)."""
    try:
        v = sp.sympify(str(texto).replace("^", "**"), locals={"pi": sp.pi, "sqrt": sp.sqrt, "E": sp.E})
    except (sp.SympifyError, TypeError):
        raise ValueError(f"'{texto}' no es un número.") from None
    if v.free_symbols or not v.is_real:
        raise ValueError(f"'{texto}' no es un número real.")
    return v


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas y devuelve texto."""
    if valor is None:
        return "no existe"
    if valor in (sp.oo, sp.S.Infinity) or (isinstance(valor, float) and valor == math.inf):
        return "+∞"
    if valor in (-sp.oo, sp.S.NegativeInfinity) or (isinstance(valor, float) and valor == -math.inf):
        return "-∞"
    v = float(valor)
    if math.isnan(v):
        return "no definida"
    if v == 0:
        return "0"
    return np.format_float_positional(v, precision=n, unique=False, fractional=False, trim="-")


def a_texto(valor):
    """Límite de sympy en texto: número exacto si es racional, ±∞ o 'no existe'."""
    if valor is None or valor is sp.nan:
        return "no existe"
    if valor.has(sp.I):
        return "no definido (fuera del dominio)"
    if valor == sp.oo:
        return "+∞"
    if valor == -sp.oo:
        return "-∞"
    if valor == sp.zoo:
        return "∞ sin signo (no existe)"
    if isinstance(valor, sp.AccumBounds):
        return "no existe (oscila)"
    return str(valor) if valor.is_Rational else f"{valor} ≈ {cifras(valor, 6)}"


def conjunto_a_texto(s):
    """Intervalos de sympy en notación de intervalos."""
    if s == sp.S.Reals:
        return "(-∞, ∞) = ℝ"
    if s == sp.S.EmptySet:
        return "∅ (vacío)"
    if isinstance(s, sp.Union):
        return " ∪ ".join(conjunto_a_texto(t) for t in sorted(s.args, key=lambda t: float(t.inf)))
    if isinstance(s, sp.Interval):
        f = lambda p: ("∞" if p == sp.oo else "-∞" if p == -sp.oo
                       else str(p) if p.is_Rational else "≈" + cifras(float(p), 4))
        return ("(" if s.left_open else "[") + f(s.inf) + ", " + f(s.sup) + (")" if s.right_open else "]")
    return str(s)


def evaluar(f, v):
    """f(v) en punto flotante; nan si f no está definida en v."""
    with np.errstate(all="ignore"):
        try:
            r = complex(f(v))
        except (ZeroDivisionError, ValueError, OverflowError):
            return math.nan
    return r.real if abs(r.imag) < 1e-12 else math.nan


def cerca(u, v, tol=1e-9):
    return abs(u - v) < tol


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


def definida_en(f, a):
    """True si la expresión f tiene valor real finito en x = a."""
    v = f.subs(x, a)
    return bool(v.is_real) and bool(v.is_finite)


def pendiente_secante(f, a, b):
    """(f(b) - f(a)) / (b - a), exacta. Rechaza a = b y puntos fuera del dominio."""
    a, b = sp.nsimplify(a), sp.nsimplify(b)
    if a == b:
        raise ValueError("a y b son iguales: con un solo punto no hay secante (Δx = 0).")
    for p in (a, b):
        if not definida_en(f, p):
            raise ValueError(f"f no está definida en x = {p}: ese punto no está en la gráfica.")
    return sp.simplify((f.subs(x, b) - f.subs(x, a)) / (b - a))


def cociente(f, a):
    """Cociente incremental (f(a + h) - f(a)) / h como expresión en h."""
    a = sp.nsimplify(a)
    if not definida_en(f, a):
        raise ValueError(f"f no está definida en x = {a}: no hay punto de tangencia.")
    return (f.subs(x, a + h) - f.subs(x, a)) / h


def lado_definido(f, a, signo):
    """True si f tiene valores reales justo a la izquierda (signo = -1) o a la derecha (signo = +1) de a."""
    v = f.subs(x, sp.nsimplify(a) + signo * sp.Rational(1, 10**6))
    return bool(v.is_real) and bool(v.is_finite)


def lado_texto(valor):
    """Derivada lateral en texto; None significa que ese lado queda fuera del dominio."""
    return "no definida (ese lado está fuera del dominio)" if valor is None else a_texto(valor)


def derivada_en(f, a):
    """(izquierda, derecha) del límite del cociente cuando h -> 0, y f'(a) o None si no existe.
    Un lateral vale None si f no está definida de ese lado de a (extremo del dominio, como √x en 0)."""
    q = cociente(f, a)
    izq = sp.limit(q, h, 0, "-") if lado_definido(f, a, -1) else None
    der = sp.limit(q, h, 0, "+") if lado_definido(f, a, +1) else None
    existe = (izq is not None and der is not None and izq.is_finite and der.is_finite
              and sp.simplify(izq - der) == 0)
    return izq, der, (sp.simplify(der) if existe else None)

## 1. Pendiente de una recta y pendiente de la secante

La pendiente de la recta que pasa por $(x_1,y_1)$ y $(x_2,y_2)$ es
$$m=\frac{\Delta y}{\Delta x}=\frac{y_2-y_1}{x_2-x_1},\qquad x_1\neq x_2.$$
En una recta da lo mismo con cualquier par de puntos. En una curva, la recta que pasa por $(a,f(a))$ y $(b,f(b))$ es una **secante** y su pendiente cambia con los puntos: es la **razón de cambio promedio** de $f$ entre $a$ y $b$.

La calculadora recibe $f$, $a$ y $b$, da la pendiente exacta y redondeada, y dibuja la curva con la secante.

In [ ]:
def calculadora_secante(f_txt, a, b, n_cifras):
    try:
        f = parsear(f_txt)
        m = pendiente_secante(f, a, b)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    fa, fb = f.subs(x, sp.nsimplify(a)), f.subs(x, sp.nsimplify(b))
    print(f"f(a) = {cifras(fa, n_cifras)}   f(b) = {cifras(fb, n_cifras)}")
    print(f"Δy/Δx = ({cifras(fb, n_cifras)} - {cifras(fa, n_cifras)}) / ({b} - {a})")
    print(f"pendiente de la secante = {m} ≈ {cifras(m, n_cifras)}  ({n_cifras} cifras significativas)")
    fn = sp.lambdify(x, f, "numpy")
    lo, hi = min(a, b), max(a, b); ancho = hi - lo
    xs = np.linspace(lo - ancho, hi + ancho, 400)
    with np.errstate(all="ignore"):
        ys = np.broadcast_to(fn(xs), xs.shape).astype(float)
    plt.figure(figsize=(5, 3.2)); plt.plot(xs, ys, color="gray", label="f")
    plt.plot(xs, float(fa) + float(m) * (xs - a), "b--", label=f"secante, m ≈ {cifras(m, 4)}")
    plt.plot([a, b], [float(fa), float(fb)], "ro"); plt.grid(True); plt.legend(fontsize=8); plt.show()


widgets.interact(calculadora_secante,
    f_txt=widgets.Text(value="x**2", description="f(x) ="),
    a=widgets.FloatText(value=1.0, description="a"), b=widgets.FloatText(value=3.0, description="b"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
PRUEBAS_1 = [
    ("recta 0.5x + 1: pendiente 1/2 con (1, 3) y con (-2, 10)",
     lambda: pendiente_secante(parsear("0.5*x + 1"), 1, 3) == sp.Rational(1, 2)
             and pendiente_secante(parsear("0.5*x + 1"), -2, 10) == sp.Rational(1, 2)),
    ("x² de 1 a 3: pendiente 4", lambda: pendiente_secante(x**2, 1, 3) == 4),
    ("x² de 1 a 2: pendiente 3", lambda: pendiente_secante(x**2, 1, 2) == 3),
    ("costo 2000 + 15q + 0.05q² de 100 a 120: 26 pesos/pieza",
     lambda: pendiente_secante(parsear("2000 + 15*x + 0.05*x**2"), 100, 120) == 26),
    ("el orden de los puntos no cambia la pendiente",
     lambda: pendiente_secante(x**3, 1, 2) == pendiente_secante(x**3, 2, 1)),
    ("a = b se rechaza con mensaje", lambda: _rechaza(lambda: pendiente_secante(x**2, 2, 2))),
    ("punto fuera del dominio se rechaza", lambda: _rechaza(lambda: pendiente_secante(sp.sqrt(x), -1, 4))),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 1).** Calcula a mano la pendiente de la secante de $y=x^3$ entre $x=1$ y $x=2$. Escribe tu resultado y compáralo con la calculadora.

In [ ]:
mi_pendiente = None       # escribe un número, por ejemplo: 2.5

ref = pendiente_secante(x**3, 1, 2)
print("La calculadora da:", ref)
print("falta tu cálculo" if mi_pendiente is None else
      ("coincide" if cerca(mi_pendiente, float(ref)) else "NO coincide: Δy = 2³ - 1³ y Δx = 2 - 1"))

## 2. Paso al límite: definición de derivada

Con $b=a+h$, la pendiente de la secante es el **cociente incremental** $\dfrac{f(a+h)-f(a)}{h}$. La derivada de $f$ en $a$ es su límite:
$$f'(a)=\lim_{h\to 0}\frac{f(a+h)-f(a)}{h},$$
si existe. Aquí $h$ es una **variable** que tiende a cero por los dos lados; un valor fijo de $h$ da solo la pendiente de una secante.

La calculadora tabula el cociente con $h=\pm10^{-1},\dots,\pm10^{-k}$ y lo compara con el límite exacto que calcula `sympy`. Con $|h|$ menor que $10^{-8}$ el redondeo de la computadora empieza a dominar, por eso el control llega hasta $k=8$.

In [ ]:
def tabla_cocientes(f, a, k=6):
    """[(h, cociente)] con h = ±10^-1, ..., ±10^-k, en punto flotante."""
    fn = sp.lambdify(x, f, "math")
    fa = evaluar(fn, float(a))
    if math.isnan(fa):
        raise ValueError(f"f no está definida en x = {a}: no hay punto de partida.")
    filas = []
    for j in range(1, k + 1):
        for hh in (10.0**-j, -10.0**-j):
            filas.append((hh, (evaluar(fn, float(a) + hh) - fa) / hh))
    return filas


def calculadora_limite(f_txt, a, k, n_cifras):
    try:
        f = parsear(f_txt)
        filas = tabla_cocientes(f, a, k)
        izq, der, d = derivada_en(f, a)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"{'h':>12}   cociente ({n_cifras} cifras significativas)")
    for hh, q in filas:
        print(f"{hh:>12g}   {cifras(q, n_cifras) if not math.isnan(q) else 'no definido'}")
    print("límite por la izquierda:", lado_texto(izq), "| por la derecha:", lado_texto(der))
    print("f'(a) =", a_texto(d) if d is not None else "no existe")
    print("Nota: k llega hasta 8 porque con |h| menor que 1e-8 el redondeo de la computadora domina el cociente.")


widgets.interact(calculadora_limite,
    f_txt=widgets.Text(value="20 + 70*exp(-0.1*x)", description="f(x) ="),
    a=widgets.FloatText(value=5.0, description="a"),
    k=widgets.IntSlider(value=4, min=1, max=8, description="k"),
    n_cifras=widgets.IntSlider(value=5, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
cafe = 20 + 70 * sp.exp(-x / 10)
PRUEBAS_2 = [
    ("x² en 1: el cociente con h = 0.001 vale 2.001 y el límite es 2",
     lambda: cerca(dict(tabla_cocientes(x**2, 1, 3))[0.001], 2.001, 1e-9) and derivada_en(x**2, 1)[2] == 2),
    ("café en t = 5: límite -7e^(-1/2) ≈ -4.2457 °C/min",
     lambda: sp.simplify(derivada_en(cafe, 5)[2] + 7 * sp.exp(-sp.Rational(1, 2))) == 0),
    ("café: secante con h = 5 vale -3.3411", lambda: cerca(float(pendiente_secante(cafe, 5, 10)), -3.341117, 1e-5)),
    ("x³ en 2 con h = 0.1: 12.61", lambda: cerca(dict(tabla_cocientes(x**3, 2, 1))[0.1], 12.61, 1e-9)),
    ("1/x en 1 con h = 0.5: -2/3", lambda: pendiente_secante(1/x, 1, sp.Rational(3, 2)) == -sp.Rational(2, 3)),
    ("|x| en 0: laterales -1 y 1, la derivada no existe",
     lambda: derivada_en(sp.Abs(x), 0)[:2] == (-1, 1) and derivada_en(sp.Abs(x), 0)[2] is None),
    ("punto fuera del dominio se rechaza", lambda: _rechaza(lambda: tabla_cocientes(sp.sqrt(x), -1))),
    ("√x en 0: izquierda fuera del dominio, derecha +∞", lambda: derivada_en(sp.sqrt(x), 0) == (None, sp.oo, None)),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 2).** Calcula a mano el cociente incremental de $f(x)=x^2$ en $a=-2$ con $h=0.1$ y con $h=-0.1$. ¿A qué número se acercan? Compara con la calculadora.

In [ ]:
mi_cociente_mas, mi_cociente_menos = None, None     # escribe dos números, por ejemplo: 1.5, 2.5

ref = dict(tabla_cocientes(x**2, -2, 1))
print("La calculadora da:", round(ref[0.1], 10), "y", round(ref[-0.1], 10), "| límite:", derivada_en(x**2, -2)[2])
print("falta tu cálculo" if mi_cociente_mas is None else
      ("coincide" if cerca(mi_cociente_mas, ref[0.1], 1e-6) and cerca(mi_cociente_menos, ref[-0.1], 1e-6)
       else "NO coincide: ((-2 + h)² - 4)/h = -4 + h"))

## 3. Primeros ejemplos: $x^2$, $x^3$, $1/x$, $\sqrt{x}$

La receta a mano tiene tres pasos: escribir el cociente $\dfrac{f(x+h)-f(x)}{h}$, simplificarlo hasta que $h$ ya no divida (expandir, restar fracciones o multiplicar por el conjugado) y, solo entonces, hacer $h\to0$.

| $f(x)$ | $x^2$ | $x^3$ | $1/x$ | $\sqrt{x}$ |
|---|---|---|---|---|
| $f'(x)$ | $2x$ | $3x^2$ | $-1/x^2$ | $\dfrac{1}{2\sqrt{x}}$ ($x>0$) |

La calculadora aplica la definición con `sympy` a la función que escribas y da $f'(x)$ y $f'(a)$. Como comprobación independiente, compara con `sp.diff`, que ya conoce las reglas de la Unidad 5.

In [ ]:
def derivada_por_definicion(f):
    """f'(x) como límite del cociente incremental (variable x)."""
    return sp.simplify(sp.limit((f.subs(x, x + h) - f) / h, h, 0))


def calculadora_definicion(f_txt, a, n_cifras):
    try:
        f = parsear(f_txt)
        d = derivada_por_definicion(f)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print("f(x)  =", f)
    if d.has(sp.nan, sp.zoo, sp.oo, -sp.oo):
        print("f'(x): sympy no obtuvo una fórmula válida para todo x; la función puede no ser derivable")
        print("       en algún punto. Revisa ese punto con las derivadas laterales de la sección 6.")
    else:
        print("f'(x) =", d, "  (por definición)")
        print("comprobación con sp.diff:", "coincide" if sp.simplify(d - sp.diff(f, x)) == 0 else "NO coincide")
    try:
        izq, der, da = derivada_en(f, a)
    except ValueError as err:
        print("En a:", err); return
    print(f"f'({a}) =", (f"{da} ≈ {cifras(da, n_cifras)}  ({n_cifras} cifras significativas)" if da is not None
                         else f"no existe (izquierda: {lado_texto(izq)}; derecha: {lado_texto(der)})"))


widgets.interact(calculadora_definicion,
    f_txt=widgets.Text(value="sqrt(x)", description="f(x) ="),
    a=widgets.FloatText(value=4.0, description="a"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
PRUEBAS_3 = [
    ("x² -> 2x", lambda: derivada_por_definicion(x**2) == 2*x),
    ("x³ -> 3x²", lambda: derivada_por_definicion(x**3) == 3*x**2),
    ("1/x -> -1/x²", lambda: sp.simplify(derivada_por_definicion(1/x) + 1/x**2) == 0),
    ("√x -> 1/(2√x)", lambda: sp.simplify(derivada_por_definicion(sp.sqrt(x)) - 1/(2*sp.sqrt(x))) == 0),
    ("3x² - 5x -> 6x - 5", lambda: sp.expand(derivada_por_definicion(3*x**2 - 5*x)) == 6*x - 5),
    ("constante 7 -> 0", lambda: derivada_por_definicion(sp.Integer(7) + 0*x) == 0),
    ("recta 4x - 1 -> 4", lambda: derivada_por_definicion(4*x - 1) == 4),
    ("√x en 4: 1/4", lambda: derivada_en(sp.sqrt(x), 4)[2] == sp.Rational(1, 4)),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 3).** Calcula a mano, con la definición, la derivada de $f(x)=2x^2+1$. Escribe tu resultado como texto (por ejemplo `"4*x"`) y compáralo con la calculadora.

In [ ]:
mi_derivada = None       # escribe un texto, por ejemplo: "3*x**2"

ref = derivada_por_definicion(2*x**2 + 1)
print("La calculadora da:", ref)
print("falta tu cálculo" if mi_derivada is None else
      ("coincide" if sp.simplify(parsear(mi_derivada) - ref) == 0 else "NO coincide: ¿el 1 se canceló al restar f(x)?"))

## 4. Notaciones $f'(x)$ y $dy/dx$; recta tangente

Si $y=f(x)$, la derivada se escribe $f'(x)$, $y'$, $\dfrac{dy}{dx}$ o $\dfrac{d}{dx}f(x)$. Su valor en $a$ es $f'(a)$ o $\left.\dfrac{dy}{dx}\right|_{x=a}$. La recta tangente en $(a,f(a))$ es
$$y=f(a)+f'(a)\,(x-a),$$
y cerca de $a$ sirve para aproximar: $f(a+\Delta x)\approx f(a)+f'(a)\,\Delta x$.

La calculadora da la ecuación de la tangente, compara la aproximación con el valor exacto en $a+\Delta x$ y dibuja las dos. Si los laterales del cociente tienden a $\pm\infty$ con el mismo signo, la tangente es vertical, $x=a$; si son distintos, no hay tangente.

In [ ]:
def tangente(f, a):
    """(m, b) de la recta tangente y = m x + b en x = a; ValueError si no existe o es vertical."""
    izq, der, m = derivada_en(f, a)
    if m is None:
        infinitos = (sp.oo, -sp.oo)
        if izq is not None and der is not None and izq == der and izq in infinitos:
            raise ValueError(f"el cociente tiende a {a_texto(der)} por los dos lados: la tangente es vertical, x = {a}.")
        if (izq is None) != (der is None):
            lado, v = ("derecha", der) if izq is None else ("izquierda", izq)
            if v in infinitos:
                raise ValueError(f"x = {a} es un extremo del dominio y el cociente por la {lado} tiende a {a_texto(v)}: "
                                 f"la tangente es vertical, x = {a}, pero f'({a}) no existe.")
            raise ValueError(f"x = {a} es un extremo del dominio: solo existe la derivada por la {lado}, {a_texto(v)}.")
        raise ValueError(f"los laterales del cociente son {lado_texto(izq)} y {lado_texto(der)}: no hay recta tangente.")
    a = sp.nsimplify(a)
    return m, sp.simplify(f.subs(x, a) - m * a)


def calculadora_tangente(f_txt, a, dx, n_cifras):
    try:
        f = parsear(f_txt)
        m, b = tangente(f, a)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    a_, dx_ = sp.nsimplify(a), sp.nsimplify(dx)
    aprox, exacto = m * (a_ + dx_) + b, f.subs(x, a_ + dx_)
    print(f"f'({a}) = dy/dx en x = {a}: {m} ≈ {cifras(m, n_cifras)}")
    print(f"tangente: y = {cifras(m, n_cifras)} x + {cifras(b, n_cifras)}   (exacta: y = {sp.expand(m*x + b)})")
    print(f"en x = {a} + {dx}: tangente {cifras(aprox, n_cifras)} | f exacta {cifras(exacto, n_cifras)} "
          f"| error {cifras(sp.N(aprox - exacto), 3)}  ({n_cifras} cifras significativas)")
    fn = sp.lambdify(x, f, "numpy")
    xs = np.linspace(float(a) - 3 * max(abs(dx), 0.5), float(a) + 3 * max(abs(dx), 0.5), 400)
    with np.errstate(all="ignore"):
        ys = np.broadcast_to(fn(xs), xs.shape).astype(float)
    plt.figure(figsize=(5, 3.2)); plt.plot(xs, ys, color="gray", label="f")
    plt.plot(xs, float(m) * xs + float(b), "b--", label="tangente")
    plt.plot([a], [float(f.subs(x, a_))], "ro"); plt.grid(True); plt.legend(fontsize=8); plt.show()


widgets.interact(calculadora_tangente,
    f_txt=widgets.Text(value="1/x", description="f(x) ="),
    a=widgets.FloatText(value=2.0, description="a"), dx=widgets.FloatText(value=0.1, description="Δx"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
PRUEBAS_4 = [
    ("x² en 3: y = 6x - 9", lambda: tangente(x**2, 3) == (6, -9)),
    ("1/x en 2: y = -x/4 + 1", lambda: tangente(1/x, 2) == (-sp.Rational(1, 4), 1)),
    ("√x en 4: y = x/4 + 1", lambda: tangente(sp.sqrt(x), 4) == (sp.Rational(1, 4), 1)),
    ("x³ en -1: y = 3x + 2", lambda: tangente(x**3, -1) == (3, 2)),
    ("Torricelli √(19.62 x) en 2: pendiente ≈ 1.5660",
     lambda: cerca(float(tangente(parsear("sqrt(19.62*x)"), 2)[0]), 1.566046, 1e-5)),
    ("|x| en 0: sin tangente (laterales -1 y 1)", lambda: _rechaza(lambda: tangente(sp.Abs(x), 0))),
    ("cbrt(x) en 0: tangente vertical, se avisa", lambda: _rechaza(lambda: tangente(parsear("cbrt(x)"), 0))),
    ("√x en 0: extremo con tangente vertical, se avisa", lambda: _rechaza(lambda: tangente(sp.sqrt(x), 0))),
]
for nombre, prueba in PRUEBAS_4:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 4).** Escribe a mano la recta tangente a $y=\sqrt{x}$ en $x=9$ en la forma $y=mx+b$. Compárala con la calculadora.

In [ ]:
mi_m, mi_b = None, None      # escribe dos números, por ejemplo: 0.5, 1

ref = tangente(sp.sqrt(x), 9)
print("La calculadora da: m =", ref[0], ", b =", ref[1])
print("falta tu cálculo" if mi_m is None else
      ("coincide" if cerca(mi_m, float(ref[0])) and cerca(mi_b, float(ref[1])) else "NO coincide: y = f(9) + f'(9)(x - 9), con f'(9) = 1/(2·3)"))

## 5. La derivada como razón de cambio

La pendiente de la secante entre $a$ y $a+h$ es la **razón de cambio promedio**; la derivada $f'(a)$ es la **razón de cambio instantánea**. Las dos tienen las mismas unidades: las de $y$ entre las de $x$ (m/s, °C/min, pesos/pieza, A/s).

La calculadora recibe $f$ (con $x$ como variable, aunque represente el tiempo), el instante $a$, el intervalo $h$ y las unidades. Da las dos razones de cambio con sus unidades y cuánto se separan.

In [ ]:
def razones(f, a, hh):
    """(promedio en [a, a + h], instantánea en a), exactas."""
    a, hh = sp.nsimplify(a), sp.nsimplify(hh)
    prom = pendiente_secante(f, a, a + hh)
    izq, der, inst = derivada_en(f, a)
    if inst is None:
        raise ValueError(f"la razón instantánea no existe en x = {a} (izquierda: {lado_texto(izq)}; derecha: {lado_texto(der)}).")
    return prom, inst


def calculadora_razon(f_txt, a, h_val, u_y, u_x, n_cifras):
    try:
        f = parsear(f_txt)
        prom, inst = razones(f, a, h_val)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    u = f"{u_y}/{u_x}" if u_y and u_x else "(sin unidades)"
    print(f"promedio en [{a}, {a + h_val}]: {cifras(prom, n_cifras)} {u}")
    print(f"instantánea en {a}: {cifras(inst, n_cifras)} {u}")
    dif = sp.N(prom - inst)
    rel = f" ({cifras(abs(dif / inst) * 100, 3)} % de la instantánea)" if inst != 0 else ""
    print(f"diferencia: {cifras(dif, 3)} {u}{rel}   ({n_cifras} cifras significativas)")


widgets.interact(calculadora_razon,
    f_txt=widgets.Text(value="0.6*x**2 - 0.1*x**3", description="f(x) ="),
    a=widgets.FloatText(value=2.0, description="a"), h_val=widgets.FloatText(value=1.0, description="h"),
    u_y=widgets.Text(value="m", description="unid. y"), u_x=widgets.Text(value="s", description="unid. x"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 5 (resultado conocido)
PRUEBAS_5 = [
    ("caída libre 4.9t² en 2: promedio con h = 1 es 24.5 m/s, instantánea 19.6 m/s",
     lambda: razones(parsear("4.9*x**2"), 2, 1) == (sp.Rational(49, 2), sp.Rational(98, 5))),
    ("eje del robot en 2: promedio con h = 1 es 1.1 m/s, instantánea 1.2 m/s",
     lambda: razones(parsear("0.6*x**2 - 0.1*x**3"), 2, 1) == (sp.Rational(11, 10), sp.Rational(6, 5))),
    ("costo en q = 100: promedio con h = 20 es 26 pesos/pieza, instantánea 25",
     lambda: razones(parsear("2000 + 15*x + 0.05*x**2"), 100, 20) == (26, 25)),
    ("con h negativo se obtiene el promedio por la izquierda: 4.9t² en 2 con h = -1 da 14.7",
     lambda: razones(parsear("4.9*x**2"), 2, -1)[0] == sp.Rational(147, 10)),
    ("|x| en 0 no tiene razón instantánea", lambda: _rechaza(lambda: razones(sp.Abs(x), 0, 1))),
]
for nombre, prueba in PRUEBAS_5:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 5).** Para $V(t)=3t^2+2$, calcula a mano la razón de cambio promedio entre $t=1$ y $t=2$ y la instantánea en $t=1$. Compara con la calculadora ($a=1$, $h=1$).

In [ ]:
mi_promedio, mi_instantanea = None, None     # escribe dos números, por ejemplo: 4, 2

ref = razones(3*x**2 + 2, 1, 1)
print("La calculadora da: promedio", ref[0], "| instantánea", ref[1])
print("falta tu cálculo" if mi_promedio is None else
      ("coincide" if cerca(mi_promedio, float(ref[0])) and cerca(mi_instantanea, float(ref[1]))
       else "NO coincide: el promedio es (V(2) - V(1))/1 y la instantánea es V'(1) = 6·1"))

## 6. Propiedades: derivable implica continua; puntos no derivables

Las **derivadas laterales** son los límites del cociente incremental con $h\to0^-$ y con $h\to0^+$. $f$ es derivable en $a$ si las dos existen, son finitas e iguales. Si $f$ es derivable en $a$, es continua en $a$; al revés no: $|x|$ es continua en $0$ y no es derivable.

Primero se revisa la continuidad; las filas esquina, cúspide y tangente vertical suponen $f$ continua en $a$.

| Caso | Laterales del cociente | Ejemplo en $x=0$ |
|---|---|---|
| discontinuidad | (no hace falta calcularlos) | escalón |
| esquina | finitos y distintos | $\lvert x\rvert$ |
| cúspide | $-\infty$ y $+\infty$ (signos opuestos) | $x^{2/3}$ |
| tangente vertical | los dos $+\infty$ o los dos $-\infty$ | $x^{1/3}$ |

En un extremo del dominio, como $\sqrt{x}$ en $0$, solo existe el cociente de un lado.

La calculadora recibe una función por partes: la regla de la izquierda vale para $x<a$ y la de la derecha para $x\ge a$, así que $f(a)$ sale de la regla de la derecha. Para una sola fórmula escribe la misma en los dos lados. Para $x^{2/3}$ escribe `cbrt(x)**2` y para $x^{1/3}$, `cbrt(x)`.

In [ ]:
def clasificar(izq_txt, der_txt, a):
    """(tipo, lateral izquierda, lateral derecha) de la función por partes en x = a."""
    fi, fd = parsear(izq_txt), parsear(der_txt)
    a = sp.nsimplify(a)
    if not definida_en(fd, a):
        raise ValueError(f"la regla de la derecha no está definida en x = {a}, así que f(a) no existe.")
    fa = fd.subs(x, a)
    if not lado_definido(fi, a, -1):            # extremo izquierdo del dominio: solo cuenta la derecha
        if sp.simplify(sp.limit(fd, x, a, "+") - fa) != 0:
            return "discontinua por la derecha: no es derivable", None, None
        qd = sp.limit((fd.subs(x, a + h) - fa) / h, h, 0, "+")
        if qd in (sp.oo, -sp.oo):
            return "extremo del dominio con tangente vertical (solo existe el lado derecho)", None, qd
        return f"extremo del dominio: solo existe la derivada por la derecha, {qd}", None, qd
    li, ld = sp.limit(fi, x, a, "-"), sp.limit(fd, x, a, "+")
    if not (li.is_finite and ld.is_finite and sp.simplify(li - fa) == 0 and sp.simplify(ld - fa) == 0):
        return "discontinua: no es derivable (derivable implica continua)", None, None
    qi = sp.limit((fi.subs(x, a + h) - fa) / h, h, 0, "-")
    qd = sp.limit((fd.subs(x, a + h) - fa) / h, h, 0, "+")
    if qi.is_finite and qd.is_finite:
        if sp.simplify(qi - qd) == 0:
            return f"derivable, f'(a) = {sp.simplify(qd)}", qi, qd
        return "esquina: laterales finitos y distintos", qi, qd
    if qi in (sp.oo, -sp.oo) and qd in (sp.oo, -sp.oo):
        return ("tangente vertical: los dos laterales con el mismo signo infinito" if qi == qd
                else "cúspide: laterales infinitos de signos opuestos"), qi, qd
    return "no derivable: un lateral finito y otro infinito", qi, qd


def calculadora_laterales(izq_txt, der_txt, a, n_cifras):
    try:
        tipo, qi, qd = clasificar(izq_txt, der_txt, a)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    if qd is not None:
        print("derivada por la izquierda:", lado_texto(qi), "| por la derecha:", a_texto(qd))
    print("veredicto:", tipo)
    fi, fd = sp.lambdify(x, parsear(izq_txt), "numpy"), sp.lambdify(x, parsear(der_txt), "numpy")
    for hh in (0.1, 0.01, 0.001):
        ci = (evaluar(fi, a - hh) - evaluar(fd, a)) / (-hh)
        cd = (evaluar(fd, a + hh) - evaluar(fd, a)) / hh
        ci, cd = (0.0 if abs(c) < 1e-12 else c for c in (ci, cd))   # ruido de punto flotante
        print(f"   h = {hh:<6} cociente izq. {cifras(ci, n_cifras):>10}   der. {cifras(cd, n_cifras):>10}")


widgets.interact(calculadora_laterales,
    izq_txt=widgets.Text(value="abs(x)", description="x < a:"),
    der_txt=widgets.Text(value="abs(x)", description="x ≥ a:"),
    a=widgets.FloatText(value=0.0, description="a"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 6 (resultado conocido)
PRUEBAS_6 = [
    ("|x| en 0: esquina con laterales -1 y 1", lambda: clasificar("abs(x)", "abs(x)", 0)[1:] == (-1, 1)),
    ("x^(2/3) en 0: cúspide", lambda: clasificar("cbrt(x)**2", "cbrt(x)**2", 0)[0].startswith("cúspide")),
    ("x^(1/3) en 0: tangente vertical", lambda: clasificar("cbrt(x)", "cbrt(x)", 0)[0].startswith("tangente vertical")),
    ("x² en 0: derivable con f'(0) = 0", lambda: clasificar("x**2", "x**2", 0)[0] == "derivable, f'(a) = 0"),
    ("x² (x < 1) y 2x - 1 (x ≥ 1): derivable con f'(1) = 2",
     lambda: clasificar("x**2", "2*x - 1", 1)[0] == "derivable, f'(a) = 2"),
    ("escalón 0 / 2 en 0: discontinua", lambda: clasificar("0*x", "2 + 0*x", 0)[0].startswith("discontinua")),
    ("rampa 0 / 400x en 0: esquina (0 y 400)", lambda: clasificar("0*x", "400*x", 0)[1:] == (0, 400)),
    ("√x en 0: extremo del dominio con tangente vertical",
     lambda: clasificar("sqrt(x)", "sqrt(x)", 0)[0].startswith("extremo del dominio con tangente vertical")),
]
for nombre, prueba in PRUEBAS_6:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6).** Para $f(x)=|x-2|$, calcula a mano las pendientes de las secantes desde $(2,0)$ con $h=0.1$ y con $h=-0.1$. ¿Es derivable en $x=2$? Compara con la calculadora (`abs(x - 2)` en los dos lados y $a=2$).

In [ ]:
mi_izq, mi_der = None, None     # escribe dos números, por ejemplo: 0, 0

ref = clasificar("abs(x - 2)", "abs(x - 2)", 2)
print("La calculadora da:", ref[1], "y", ref[2], "->", ref[0])
print("falta tu cálculo" if mi_izq is None else
      ("coincide" if cerca(mi_izq, float(ref[1])) and cerca(mi_der, float(ref[2])) else "NO coincide: |±0.1|/(±0.1)"))

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3 + PRUEBAS_4 + PRUEBAS_5 + PRUEBAS_6
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")